In [8]:
# Install all required packages
!pip install -q gradio transformers torch accelerate sentencepiece

import gradio as gr
import json
import re
import random
from datetime import datetime
from typing import Tuple, Dict, List
import torch

print("✅ All packages installed!")

✅ All packages installed!


In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading IBM Granite 3.3 2B model... (this takes 2-5 minutes)")

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "ibm-granite/granite-3.3-2b-instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

def ask_granite(prompt: str, max_tokens: int = 300) -> str:
    """Send a prompt to IBM Granite and get a response."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return response.strip()

# Quick test
test = ask_granite("Say: HTML Quick-Styler is ready!", max_tokens=20)
print(f"✅ Model ready! Test: {test}")

Loading IBM Granite 3.3 2B model... (this takes 2-5 minutes)


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

✅ Model ready! Test: HTML Quick-Styler is ready for your use! Let's enhance your HTML code with ease


In [10]:
class ColorPaletteGenerator:
    """Generate visually cohesive color palettes for 8 design styles."""

    @staticmethod
    def generate_palette(style: str = "modern") -> Dict[str, str]:
        palettes = {
            "modern": {
                "primary":        "#667eea",
                "secondary":      "#764ba2",
                "accent":         "#f093fb",
                "background":     "#0f172a",
                "surface":        "#1e293b",
                "text_primary":   "#f1f5f9",
                "text_secondary": "#94a3b8",
                "border":         "#334155"
            },
            "vibrant": {
                "primary":        "#ff6b6b",
                "secondary":      "#ff8e72",
                "accent":         "#ff5726",
                "background":     "#1a0a00",
                "surface":        "#2d1500",
                "text_primary":   "#fff5f0",
                "text_secondary": "#ffa07a",
                "border":         "#4d2010"
            },
            "ocean": {
                "primary":        "#4facfe",
                "secondary":      "#00f2fe",
                "accent":         "#0072ff",
                "background":     "#001020",
                "surface":        "#001e35",
                "text_primary":   "#e0f7ff",
                "text_secondary": "#7ecbf7",
                "border":         "#0a3050"
            },
            "forest": {
                "primary":        "#43e97b",
                "secondary":      "#38f9d7",
                "accent":         "#0ba360",
                "background":     "#021208",
                "surface":        "#051f0e",
                "text_primary":   "#e8fff2",
                "text_secondary": "#7ed8a4",
                "border":         "#0d3d1e"
            },
            "sunset": {
                "primary":        "#f7971e",
                "secondary":      "#ffd200",
                "accent":         "#f64f59",
                "background":     "#1a0800",
                "surface":        "#2d1500",
                "text_primary":   "#fff8e1",
                "text_secondary": "#ffc87a",
                "border":         "#4d2800"
            },
            "luxury": {
                "primary":        "#c9a96e",
                "secondary":      "#f5d08a",
                "accent":         "#8b6914",
                "background":     "#0d0a00",
                "surface":        "#1a1400",
                "text_primary":   "#fffdf0",
                "text_secondary": "#d4b896",
                "border":         "#3d3000"
            },
            "creative": {
                "primary":        "#9b59b6",
                "secondary":      "#8e44ad",
                "accent":         "#e74c3c",
                "background":     "#0f0015",
                "surface":        "#1a0025",
                "text_primary":   "#fdf0ff",
                "text_secondary": "#c39bd3",
                "border":         "#3d0050"
            },
            "minimal": {
                "primary":        "#2c3e50",
                "secondary":      "#34495e",
                "accent":         "#3498db",
                "background":     "#0a0d10",
                "surface":        "#141820",
                "text_primary":   "#ecf0f1",
                "text_secondary": "#95a5a6",
                "border":         "#2c3e50"
            }
        }
        return palettes.get(style.lower(), palettes["modern"])

    @staticmethod
    def get_color_suggestions(style: str) -> str:
        palette = ColorPaletteGenerator.generate_palette(style)
        result = f"### 🎨 Palette for **{style}** style:\n\n"
        for key, value in palette.items():
            result += f"- **{key.replace('_',' ').title()}:** `{value}`\n"
        return result

print("✅ ColorPaletteGenerator ready with 8 palettes!")

✅ ColorPaletteGenerator ready with 8 palettes!


In [11]:
class HTMLPageBuilder:
    """Build complete styled HTML pages from user descriptions."""

    @staticmethod
    def extract_sections(description: str) -> List[str]:
        """Detect which sections are needed from keywords."""
        sections = []
        section_keywords = {
            "hero":         ["hero", "banner", "landing", "welcome", "header"],
            "features":     ["features", "services", "benefits", "offerings"],
            "about":        ["about", "team", "company", "mission", "vision"],
            "portfolio":    ["portfolio", "gallery", "work", "projects", "showcase"],
            "testimonials": ["testimonials", "testimonial", "reviews", "feedback", "clients"],
            "pricing":      ["pricing", "plans", "cost", "packages", "tiers"],
            "contact":      ["contact", "form", "reach", "inquiry", "get in touch"],
            "footer":       ["footer", "bottom", "links", "social"]
        }
        desc_lower = description.lower()
        for section_type, keywords in section_keywords.items():
            for keyword in keywords:
                if keyword in desc_lower:
                    if section_type not in sections:
                        sections.append(section_type)
                    break

        # Always include hero and footer if nothing matched
        if not sections:
            sections = ["hero", "features", "contact", "footer"]
        if "footer" not in sections:
            sections.append("footer")

        return sections

    @staticmethod
    def generate_meta_tags(title: str, description: str) -> str:
        return f"""
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description" content="{description[:160]}">
    <meta name="theme-color" content="#667eea">
    <meta property="og:title" content="{title}">
    <meta property="og:description" content="{description[:200]}">
    <meta property="og:type" content="website">
    <meta name="twitter:card" content="summary_large_image">
    <meta name="author" content="HTML Quick-Styler">
    <title>{title}</title>"""

    @staticmethod
    def generate_css(palette: Dict) -> str:
        return f"""
    <style>
      :root {{
        --primary:        {palette['primary']};
        --secondary:      {palette['secondary']};
        --accent:         {palette['accent']};
        --bg:             {palette['background']};
        --surface:        {palette['surface']};
        --text:           {palette['text_primary']};
        --muted:          {palette['text_secondary']};
        --border:         {palette['border']};
      }}
      * {{ margin:0; padding:0; box-sizing:border-box; }}
      html {{ scroll-behavior: smooth; }}
      body {{
        font-family: 'Segoe UI', system-ui, sans-serif;
        background: var(--bg);
        color: var(--text);
        line-height: 1.6;
      }}
      /* NAV */
      nav {{
        position: sticky; top:0; z-index:100;
        background: rgba(0,0,0,0.7);
        backdrop-filter: blur(16px);
        border-bottom: 1px solid var(--border);
        padding: 0 2rem;
        display: flex; align-items:center; justify-content:space-between;
        height: 64px;
      }}
      .logo {{
        font-size:1.3rem; font-weight:800;
        color: var(--primary);
      }}
      nav ul {{ list-style:none; display:flex; gap:2rem; }}
      nav a {{ color:var(--muted); text-decoration:none; transition:color .2s; }}
      nav a:hover {{ color:var(--text); }}
      .nav-cta {{
        background: var(--primary); color:#fff !important;
        padding:.4rem 1.2rem; border-radius:8px; font-weight:600;
      }}
      /* HERO */
      .hero {{
        min-height: 90vh;
        display:flex; flex-direction:column; align-items:center; justify-content:center;
        text-align:center; padding:4rem 2rem;
        background: linear-gradient(135deg, var(--bg) 0%, var(--surface) 100%);
        position:relative; overflow:hidden;
      }}
      .hero::before {{
        content:''; position:absolute; inset:0;
        background: radial-gradient(ellipse 70% 50% at 50% 0%, color-mix(in srgb, var(--primary) 20%, transparent), transparent);
      }}
      .hero h1 {{
        font-size: clamp(2.5rem, 6vw, 4.5rem);
        font-weight:900; line-height:1.1;
        background: linear-gradient(135deg, var(--text) 40%, var(--accent));
        -webkit-background-clip:text; -webkit-text-fill-color:transparent;
        margin-bottom:1.5rem;
      }}
      .hero p {{ color:var(--muted); font-size:1.15rem; max-width:600px; margin-bottom:2.5rem; }}
      .btn {{
        display:inline-block; padding:.8rem 2rem; border-radius:10px;
        font-weight:700; font-size:1rem; text-decoration:none; border:none;
        cursor:pointer; transition:transform .2s, box-shadow .2s;
      }}
      .btn:hover {{ transform:translateY(-3px); box-shadow:0 12px 30px rgba(0,0,0,.3); }}
      .btn-primary {{ background: linear-gradient(135deg, var(--primary), var(--secondary)); color:#fff; }}
      .btn-outline {{ border:2px solid var(--border); color:var(--text); background:transparent; }}
      .btn-outline:hover {{ border-color:var(--primary); }}
      /* SECTIONS */
      section {{ padding:5rem 2rem; }}
      .container {{ max-width:1100px; margin:0 auto; }}
      .section-title {{
        font-size:clamp(1.8rem,4vw,2.5rem); font-weight:800; margin-bottom:.75rem;
      }}
      .section-sub {{ color:var(--muted); font-size:1rem; margin-bottom:3rem; max-width:550px; }}
      .tag {{
        display:inline-block; font-size:.75rem; font-weight:700; letter-spacing:.12em;
        text-transform:uppercase; color:var(--primary); margin-bottom:.75rem;
      }}
      /* GRID CARDS */
      .grid-3 {{ display:grid; grid-template-columns:repeat(auto-fit, minmax(280px,1fr)); gap:1.5rem; }}
      .card {{
        background:var(--surface); border:1px solid var(--border);
        border-radius:16px; padding:2rem;
        transition:transform .3s, border-color .3s, box-shadow .3s;
      }}
      .card:hover {{
        transform:translateY(-4px); border-color:var(--primary);
        box-shadow:0 16px 40px rgba(0,0,0,.3);
      }}
      .card-icon {{ font-size:2.5rem; margin-bottom:1rem; }}
      .card h3 {{ font-size:1.1rem; font-weight:700; margin-bottom:.5rem; }}
      .card p {{ color:var(--muted); font-size:.9rem; }}
      /* PORTFOLIO GALLERY */
      .gallery {{ display:grid; grid-template-columns:repeat(auto-fill,minmax(250px,1fr)); gap:1rem; }}
      .gallery-item {{
        background:var(--surface); border:1px solid var(--border);
        border-radius:12px; overflow:hidden; aspect-ratio:4/3;
        display:flex; align-items:center; justify-content:center;
        font-size:3rem; transition:transform .3s;
        cursor:pointer;
      }}
      .gallery-item:hover {{ transform:scale(1.03); border-color:var(--primary); }}
      /* TESTIMONIALS */
      .testimonial-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(300px,1fr)); gap:1.5rem; }}
      .testimonial {{
        background:var(--surface); border:1px solid var(--border);
        border-radius:16px; padding:2rem; position:relative;
      }}
      .testimonial::before {{
        content:'"'; position:absolute; top:1rem; right:1.5rem;
        font-size:4rem; color:var(--primary); opacity:.3; line-height:1;
      }}
      .testimonial p {{ color:var(--muted); font-size:.95rem; margin-bottom:1.25rem; font-style:italic; }}
      .testimonial-author {{ font-weight:700; font-size:.9rem; color:var(--text); }}
      .testimonial-role {{ font-size:.8rem; color:var(--muted); }}
      .stars {{ color:var(--accent); margin-bottom:.75rem; }}
      /* PRICING */
      .pricing-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:1.5rem; }}
      .pricing-card {{
        background:var(--surface); border:2px solid var(--border);
        border-radius:16px; padding:2rem; text-align:center;
        transition:transform .3s, border-color .3s;
      }}
      .pricing-card.featured {{ border-color:var(--primary); position:relative; }}
      .pricing-card.featured::before {{
        content:'Most Popular'; position:absolute; top:-14px; left:50%; transform:translateX(-50%);
        background:var(--primary); color:#fff; font-size:.75rem; font-weight:700;
        padding:.3rem 1rem; border-radius:100px;
      }}
      .pricing-card:hover {{ transform:translateY(-4px); border-color:var(--primary); }}
      .price {{ font-size:3rem; font-weight:900; color:var(--primary); }}
      .price span {{ font-size:1rem; color:var(--muted); font-weight:400; }}
      .price-features {{ list-style:none; margin:1.5rem 0; text-align:left; display:flex; flex-direction:column; gap:.75rem; }}
      .price-features li {{ color:var(--muted); font-size:.9rem; display:flex; align-items:center; gap:.75rem; }}
      .price-features li::before {{ content:'✓'; color:var(--primary); font-weight:700; }}
      /* CONTACT */
      .contact-wrapper {{ display:grid; grid-template-columns:1fr 1.5fr; gap:3rem; align-items:start; }}
      .contact-info h3 {{ font-size:1.5rem; font-weight:800; margin-bottom:1rem; }}
      .contact-info p {{ color:var(--muted); margin-bottom:1.5rem; }}
      .contact-detail {{ display:flex; gap:.75rem; align-items:center; margin-bottom:1rem; color:var(--muted); font-size:.9rem; }}
      .contact-detail span {{ font-size:1.25rem; }}
      form {{ display:flex; flex-direction:column; gap:1rem; }}
      input, textarea, select {{
        background:var(--surface); border:1px solid var(--border);
        border-radius:10px; padding:.9rem 1.25rem; color:var(--text);
        font-family:inherit; font-size:.95rem; width:100%;
        transition:border-color .2s;
      }}
      input:focus, textarea:focus, select:focus {{
        outline:none; border-color:var(--primary);
      }}
      textarea {{ min-height:130px; resize:vertical; }}
      /* TEAM */
      .team-grid {{ display:grid; grid-template-columns:repeat(auto-fill,minmax(200px,1fr)); gap:1.5rem; }}
      .team-card {{
        background:var(--surface); border:1px solid var(--border);
        border-radius:16px; padding:2rem; text-align:center;
        transition:transform .3s, border-color .3s;
      }}
      .team-card:hover {{ transform:translateY(-4px); border-color:var(--primary); }}
      .avatar {{
        width:80px; height:80px; border-radius:50%;
        background: linear-gradient(135deg, var(--primary), var(--secondary));
        display:flex; align-items:center; justify-content:center;
        font-size:2rem; margin:0 auto 1rem;
      }}
      .team-card h3 {{ font-size:1rem; font-weight:700; }}
      .team-card p {{ color:var(--muted); font-size:.85rem; }}
      /* FOOTER */
      footer {{
        background:var(--surface); border-top:1px solid var(--border);
        padding:3rem 2rem; text-align:center;
      }}
      .footer-links {{ display:flex; justify-content:center; gap:2rem; flex-wrap:wrap; margin-bottom:1.5rem; }}
      .footer-links a {{ color:var(--muted); text-decoration:none; font-size:.9rem; transition:color .2s; }}
      .footer-links a:hover {{ color:var(--primary); }}
      .footer-copy {{ color:var(--muted); font-size:.85rem; }}
      /* RESPONSIVE */
      @media(max-width:768px) {{
        nav ul {{ display:none; }}
        .contact-wrapper {{ grid-template-columns:1fr; }}
        .hero h1 {{ font-size:2.2rem; }}
      }}
      @keyframes fadeUp {{
        from {{ opacity:0; transform:translateY(20px); }}
        to   {{ opacity:1; transform:translateY(0); }}
      }}
      .hero > * {{ animation: fadeUp .7s ease forwards; }}
    </style>"""

    @staticmethod
    def generate_section(section_type: str, title: str, palette: Dict, ai_content: str = "") -> str:
        sections = {
            "hero": f"""
    <nav>
      <div class="logo">⚡ {title}</div>
      <ul>
        <li><a href="#features">Features</a></li>
        <li><a href="#about">About</a></li>
        <li><a href="#portfolio">Work</a></li>
        <li><a href="#contact" class="nav-cta">Contact</a></li>
      </ul>
    </nav>
    <section class="hero">
      <p class="tag">✦ AI-Generated Website</p>
      <h1>{title}</h1>
      <p>{ai_content if ai_content else 'Crafting extraordinary digital experiences with creativity, precision, and passion for outstanding design.'}</p>
      <div style="display:flex;gap:1rem;flex-wrap:wrap;justify-content:center;">
        <a href="#portfolio" class="btn btn-primary">View My Work</a>
        <a href="#contact" class="btn btn-outline">Get In Touch</a>
      </div>
    </section>""",

            "features": f"""
    <section id="features" style="background:var(--surface);">
      <div class="container">
        <p class="tag">✦ What I Offer</p>
        <h2 class="section-title">Core Features</h2>
        <p class="section-sub">Delivering exceptional results through expertise and innovation.</p>
        <div class="grid-3">
          <div class="card"><div class="card-icon">🎨</div><h3>Creative Design</h3><p>Visually stunning interfaces that captivate and engage your audience from first glance.</p></div>
          <div class="card"><div class="card-icon">⚡</div><h3>Fast Delivery</h3><p>Rapid development cycles without compromising on quality or attention to detail.</p></div>
          <div class="card"><div class="card-icon">📱</div><h3>Fully Responsive</h3><p>Perfect experience across all devices — desktop, tablet, and mobile.</p></div>
          <div class="card"><div class="card-icon">🔍</div><h3>SEO Optimized</h3><p>Built with best practices to maximize visibility and organic search performance.</p></div>
          <div class="card"><div class="card-icon">🛡️</div><h3>Secure & Reliable</h3><p>Robust architecture ensuring your site is always fast, safe, and dependable.</p></div>
          <div class="card"><div class="card-icon">🤖</div><h3>AI-Powered</h3><p>Leveraging IBM Granite AI to generate intelligent, context-aware content.</p></div>
        </div>
      </div>
    </section>""",

            "about": f"""
    <section id="about">
      <div class="container">
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:4rem;align-items:center;">
          <div>
            <p class="tag">✦ About Me</p>
            <h2 class="section-title">Passion Meets Expertise</h2>
            <p style="color:var(--muted);margin-bottom:1.5rem;line-height:1.8;">I'm a creative professional with over 5 years of experience crafting digital experiences that leave lasting impressions. My approach combines artistic sensibility with technical excellence.</p>
            <p style="color:var(--muted);margin-bottom:2rem;line-height:1.8;">From concept to deployment, I bring ideas to life with precision, creativity, and a deep understanding of user experience and modern design principles.</p>
            <a href="#contact" class="btn btn-primary">Let's Work Together</a>
          </div>
          <div style="display:grid;grid-template-columns:1fr 1fr;gap:1rem;">
            <div class="card" style="text-align:center;"><div style="font-size:2.5rem;font-weight:900;color:var(--primary);">50+</div><p>Projects Delivered</p></div>
            <div class="card" style="text-align:center;"><div style="font-size:2.5rem;font-weight:900;color:var(--primary);">98%</div><p>Client Satisfaction</p></div>
            <div class="card" style="text-align:center;"><div style="font-size:2.5rem;font-weight:900;color:var(--primary);">5yr</div><p>Experience</p></div>
            <div class="card" style="text-align:center;"><div style="font-size:2.5rem;font-weight:900;color:var(--primary);">24/7</div><p>Support</p></div>
          </div>
        </div>
      </div>
    </section>""",

            "portfolio": f"""
    <section id="portfolio" style="background:var(--surface);">
      <div class="container">
        <p class="tag">✦ My Work</p>
        <h2 class="section-title">Portfolio Gallery</h2>
        <p class="section-sub">A selection of projects showcasing diverse skills and creative solutions.</p>
        <div class="gallery">
          <div class="gallery-item" style="background:linear-gradient(135deg,var(--primary),var(--secondary));">🎨</div>
          <div class="gallery-item" style="background:linear-gradient(135deg,var(--secondary),var(--accent));">📱</div>
          <div class="gallery-item" style="background:linear-gradient(135deg,var(--accent),var(--primary));">💻</div>
          <div class="gallery-item" style="background:linear-gradient(135deg,var(--primary),var(--accent));">🌐</div>
          <div class="gallery-item" style="background:linear-gradient(135deg,var(--secondary),var(--primary));">⚡</div>
          <div class="gallery-item" style="background:linear-gradient(135deg,var(--accent),var(--secondary));">🚀</div>
        </div>
      </div>
    </section>""",

            "testimonials": f"""
    <section id="testimonials">
      <div class="container">
        <p class="tag">✦ Client Feedback</p>
        <h2 class="section-title">What Clients Say</h2>
        <p class="section-sub">Real feedback from real clients who trusted me with their vision.</p>
        <div class="testimonial-grid">
          <div class="testimonial"><div class="stars">★★★★★</div><p>Absolutely incredible work! The attention to detail and creativity exceeded all our expectations. Delivered on time and beyond scope.</p><div class="testimonial-author">Sarah Johnson</div><div class="testimonial-role">CEO, TechStart Inc.</div></div>
          <div class="testimonial"><div class="stars">★★★★★</div><p>Professional, talented, and easy to work with. The final product was stunning and our conversions increased by 40% within the first month.</p><div class="testimonial-author">Marcus Williams</div><div class="testimonial-role">Founder, GrowthLab</div></div>
          <div class="testimonial"><div class="stars">★★★★★</div><p>Outstanding quality and incredible speed. I highly recommend to anyone looking for a world-class digital presence.</p><div class="testimonial-author">Priya Patel</div><div class="testimonial-role">Marketing Director</div></div>
        </div>
      </div>
    </section>""",

            "pricing": f"""
    <section id="pricing" style="background:var(--surface);">
      <div class="container">
        <p class="tag">✦ Pricing Plans</p>
        <h2 class="section-title">Simple, Transparent Pricing</h2>
        <p class="section-sub">Choose the plan that fits your needs. No hidden fees, ever.</p>
        <div class="pricing-grid">
          <div class="pricing-card">
            <h3>Starter</h3>
            <div class="price">$29<span>/mo</span></div>
            <ul class="price-features">
              <li>5 AI Generated Pages</li>
              <li>3 Color Palettes</li>
              <li>Basic Support</li>
              <li>Export HTML/CSS</li>
            </ul>
            <a href="#contact" class="btn btn-outline" style="width:100%;text-align:center;">Get Started</a>
          </div>
          <div class="pricing-card featured">
            <h3>Professional</h3>
            <div class="price">$79<span>/mo</span></div>
            <ul class="price-features">
              <li>Unlimited Pages</li>
              <li>All 8 Palettes</li>
              <li>Priority Support</li>
              <li>IBM Granite AI Access</li>
              <li>Live Preview</li>
            </ul>
            <a href="#contact" class="btn btn-primary" style="width:100%;text-align:center;">Get Started</a>
          </div>
          <div class="pricing-card">
            <h3>Enterprise</h3>
            <div class="price">$199<span>/mo</span></div>
            <ul class="price-features">
              <li>Everything in Pro</li>
              <li>Custom AI Training</li>
              <li>Dedicated Support</li>
              <li>API Access</li>
              <li>White Labeling</li>
            </ul>
            <a href="#contact" class="btn btn-outline" style="width:100%;text-align:center;">Contact Us</a>
          </div>
        </div>
      </div>
    </section>""",

            "contact": f"""
    <section id="contact">
      <div class="container">
        <p class="tag">✦ Get In Touch</p>
        <h2 class="section-title">Let's Work Together</h2>
        <div class="contact-wrapper">
          <div class="contact-info">
            <h3>Ready to Start?</h3>
            <p style="color:var(--muted);">Have a project in mind? I'd love to hear about it. Send me a message and let's create something amazing together.</p>
            <div class="contact-detail"><span>📧</span> hello@example.com</div>
            <div class="contact-detail"><span>📞</span> +1 (555) 123-4567</div>
            <div class="contact-detail"><span>📍</span> San Francisco, CA</div>
            <div class="contact-detail"><span>🕐</span> Mon–Fri, 9am–6pm PST</div>
          </div>
          <form onsubmit="alert('Thank you! We will be in touch soon.'); return false;">
            <div style="display:grid;grid-template-columns:1fr 1fr;gap:1rem;">
              <input type="text" placeholder="Your Name" required>
              <input type="email" placeholder="Email Address" required>
            </div>
            <input type="text" placeholder="Subject">
            <textarea placeholder="Tell me about your project..."></textarea>
            <button type="submit" class="btn btn-primary">Send Message →</button>
          </form>
        </div>
      </div>
    </section>""",

            "footer": f"""
    <footer>
      <div style="font-size:1.5rem;font-weight:800;color:var(--primary);margin-bottom:1rem;">⚡ {title}</div>
      <div class="footer-links">
        <a href="#hero">Home</a>
        <a href="#features">Features</a>
        <a href="#about">About</a>
        <a href="#portfolio">Work</a>
        <a href="#contact">Contact</a>
        <a href="#">Privacy Policy</a>
        <a href="#">Terms of Service</a>
      </div>
      <p class="footer-copy">© {datetime.now().year} {title} · Built with HTML Quick-Styler & IBM Granite AI</p>
    </footer>"""
        }
        return sections.get(section_type, "")


print("✅ HTMLPageBuilder ready with all section templates!")

✅ HTMLPageBuilder ready with all section templates!


In [12]:
class HTMLGenerationEngine:
    """Orchestrates the full HTML generation pipeline."""

    @staticmethod
    def generate_page(title: str, description: str, style: str = "modern") -> Tuple[str, str]:
        try:
            # Step 1: Get color palette
            palette = ColorPaletteGenerator.generate_palette(style)

            # Step 2: Detect sections from description
            sections = HTMLPageBuilder.extract_sections(description)

            # Step 3: Generate meta tags
            meta_tags = HTMLPageBuilder.generate_meta_tags(title, description)

            # Step 4: Generate CSS
            css = HTMLPageBuilder.generate_css(palette)

            # Step 5: Ask Granite AI for hero tagline
            ai_prompt = f"Write a single compelling tagline (1-2 sentences, max 25 words) for a website called '{title}'. Description: {description}"
            try:
                ai_content = ask_granite(ai_prompt, max_tokens=60)
            except:
                ai_content = f"Professional excellence and innovation — bringing {title} to life with cutting-edge design."

            # Step 6: Assemble all sections
            sections_html = ""
            for section in sections:
                sections_html += HTMLPageBuilder.generate_section(section, title, palette, ai_content)

            # Step 7: Build final complete HTML
            html = f"""<!DOCTYPE html>
<html lang="en">
<head>
{meta_tags}
{css}
</head>
<body>
{sections_html}
</body>
</html>"""

            status = f"✅ Page generated! Sections: {', '.join(sections)} | Style: {style} | AI: IBM Granite 3.3 2B"
            return html, status

        except Exception as e:
            return f"<!-- Error: {str(e)} -->", f"❌ Error: {str(e)}"


print("✅ HTMLGenerationEngine ready!")

✅ HTMLGenerationEngine ready!


In [13]:
def build_gradio_app():
    with gr.Blocks(
        title="⚡ HTML Quick-Styler",
        theme=gr.themes.Base(
            primary_hue="violet",
            neutral_hue="slate",
        ),
        css="""
            .gradio-container { max-width: 1200px !important; }
            .tab-nav { font-weight: 600 !important; }
        """
    ) as demo:

        gr.Markdown("""
        # ⚡ HTML Quick-Styler
        ### AI-Powered HTML Page Generator with IBM Granite 3.3 2B
        *Generate production-ready HTML/CSS pages in seconds*
        """)

        with gr.Tabs():

            # ──────────────────────────────────────
            # TAB 1: GENERATE PAGE
            # ──────────────────────────────────────
            with gr.TabItem("⚡ Generate Page"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### 📋 Page Details")
                        page_title = gr.Textbox(
                            label="Page Title",
                            placeholder="e.g. Modern Portfolio Website",
                            lines=1
                        )
                        page_description = gr.Textbox(
                            label="Page Description & Sections",
                            placeholder="e.g. Hero section with welcome message, features list with 3 items, about section, portfolio gallery, testimonials, contact form, and footer",
                            lines=5
                        )
                        color_style = gr.Dropdown(
                            choices=["modern", "vibrant", "ocean", "forest", "sunset", "luxury", "creative", "minimal"],
                            value="modern",
                            label="🎨 Color Style"
                        )
                        generate_btn = gr.Button("✨ Generate HTML", variant="primary", size="lg")
                        status_msg = gr.Textbox(label="Status", interactive=False, lines=2)

                    with gr.Column(scale=2):
                        gr.Markdown("### 💻 Generated HTML/CSS")
                        html_output = gr.Code(
                            language="html",
                            label="Generated HTML",
                            lines=30,
                            interactive=False
                        )

                generate_btn.click(
                    fn=HTMLGenerationEngine.generate_page,
                    inputs=[page_title, page_description, color_style],
                    outputs=[html_output, status_msg]
                )

            # ──────────────────────────────────────
            # TAB 2: LIVE PREVIEW
            # ──────────────────────────────────────
            with gr.TabItem("👁️ Live Preview"):
                gr.Markdown("### 🖥️ Preview Generated Page")
                preview_btn = gr.Button("🔄 Load Preview", variant="primary", size="lg")
                preview_output = gr.HTML(
                    value="<div style='text-align:center;padding:3rem;color:#666;'>Click 'Load Preview' after generating a page.</div>"
                )

                def show_preview(html_text):
                    if not html_text or not html_text.strip():
                        return "<div style='text-align:center;padding:3rem;color:#ef4444;'>⚠️ No HTML generated yet. Go to Generate Page tab first!</div>"
                    return html_text

                preview_btn.click(
                    fn=show_preview,
                    inputs=[html_output],
                    outputs=[preview_output]
                )

            # ──────────────────────────────────────
            # TAB 3: COLOR PALETTES
            # ──────────────────────────────────────
            with gr.TabItem("🎨 Color Palettes"):
                gr.Markdown("### 🌈 Explore Color Schemes")
                with gr.Row():
                    style_selector = gr.Radio(
                        choices=["modern", "vibrant", "ocean", "forest", "sunset", "luxury", "creative", "minimal"],
                        value="modern",
                        label="Choose Style"
                    )
                palette_btn = gr.Button("🎨 Get Palette", variant="primary")
                palette_display = gr.Markdown()

                palette_btn.click(
                    fn=ColorPaletteGenerator.get_color_suggestions,
                    inputs=[style_selector],
                    outputs=[palette_display]
                )

            # ──────────────────────────────────────
            # TAB 4: TEMPLATES
            # ──────────────────────────────────────
            with gr.TabItem("📋 Templates"):
                gr.Markdown("### ⚡ Quick Templates")
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("""
                        #### 🎨 Portfolio Website
                        `Hero, Portfolio Gallery, About, Testimonials, Contact, Footer`
                        """)
                        portfolio_btn = gr.Button("Use Template", size="sm")

                    with gr.Column():
                        gr.Markdown("""
                        #### 🚀 SaaS Landing Page
                        `Hero, Features, Pricing, Testimonials, CTA, Footer`
                        """)
                        saas_btn = gr.Button("Use Template", size="sm")

                    with gr.Column():
                        gr.Markdown("""
                        #### 🏢 Corporate Website
                        `Header, About, Services, Team, Contact, Footer`
                        """)
                        corporate_btn = gr.Button("Use Template", size="sm")

                def load_portfolio():
                    return (
                        "Modern Portfolio Website",
                        "Hero section with welcome message, portfolio gallery with 6 items, about section with bio, testimonials from 3 clients, contact form with validation, footer with social media links",
                        "modern"
                    )

                def load_saas():
                    return (
                        "LaunchPad SaaS",
                        "Hero banner with value proposition, features showcase with icons, pricing plans comparison table, customer testimonials, call to action buttons, footer with links",
                        "vibrant"
                    )

                def load_corporate():
                    return (
                        "Nexus Corporation",
                        "Hero with company overview, about section with mission and vision, services grid, team members profiles, contact form with inquiry, footer with privacy policy",
                        "minimal"
                    )

                portfolio_btn.click(load_portfolio, outputs=[page_title, page_description, color_style])
                saas_btn.click(load_saas, outputs=[page_title, page_description, color_style])
                corporate_btn.click(load_corporate, outputs=[page_title, page_description, color_style])

                gr.Markdown("*After clicking a template, go to the **Generate Page** tab and click Generate!*")

            # ──────────────────────────────────────
            # TAB 5: GUIDE
            # ──────────────────────────────────────
            with gr.TabItem("📖 Guide"):
                gr.Markdown("""
                # 📖 HTML Quick-Styler Guide

                ## How It Works

                ### Step 1: Enter Details
                - **Title:** Your page name
                - **Description:** List the sections you want
                - **Color Style:** Pick a theme

                ### Step 2: Automatic Section Detection
                The AI recognizes these keywords in your description:
                | Keyword | Section Generated |
                |---------|------------------|
                | `hero` / `banner` | Hero Section |
                | `features` / `services` | Features Grid |
                | `about` / `team` | About Section |
                | `portfolio` / `gallery` | Portfolio Gallery |
                | `testimonials` / `reviews` | Testimonials |
                | `pricing` / `plans` | Pricing Table |
                | `contact` / `form` | Contact Form |
                | `footer` | Footer |

                ### Step 3: AI Generation
                IBM Granite 3.3 2B analyzes your input and generates:
                - Semantic HTML5 structure
                - Custom CSS with your chosen palette
                - AI-written tagline for hero section
                - Responsive layouts with mobile support
                - SEO meta tags (Open Graph, Twitter Card)

                ### Step 4: Preview & Export
                - Switch to **Live Preview** tab to see your page
                - Copy the HTML code from the output box
                - Save as `.html` and open in any browser!

                ## 🎨 Color Palettes
                | Style | Primary | Best For |
                |-------|---------|----------|
                | Modern | `#667eea` Purple | Tech, SaaS |
                | Vibrant | `#ff6b6b` Red | Creative, Portfolio |
                | Ocean | `#4facfe` Blue | Corporate, Finance |
                | Forest | `#43e97b` Green | Eco, Health |
                | Sunset | `#f7971e` Orange | Food, Travel |
                | Luxury | `#c9a96e` Gold | Fashion, Premium |
                | Creative | `#9b59b6` Violet | Art, Design |
                | Minimal | `#2c3e50` Dark | Professional |
                """)

    return demo


# Build the app
demo = build_gradio_app()
print("✅ Gradio interface built with 5 tabs!")

/tmp/ipykernel_346/2363585350.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_346/2363585350.py:2: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


✅ Gradio interface built with 5 tabs!


In [14]:
# Launch with public sharing enabled
print("\n🚀 Launching HTML Quick-Styler Advanced...")
print("=" * 50)

demo.launch(
    share=True,           # Creates a public gradio.live URL
    show_error=True,
    show_api=False,
    quiet=False
)

# You will see output like:
# ✅ Running on public URL: https://xxxxxxxx.gradio.live
# Share that URL with anyone!

/tmp/ipykernel_346/496793794.py:5: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  demo.launch(



🚀 Launching HTML Quick-Styler Advanced...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://01c272daa84366b5e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
